# Findings

- MRO logic : How `TemporalBatchMixin` executes `forward`.
- `einops` : handling 5D tensors. 

In [23]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 


TemporalBatchMixin.forward runs because ResNet5 has no forward, so the MRO finds the mixin's. ✓
Inside it, self._forward resolves via the MRO to ResNet5._forward (the subclass's version shadows the mixin's). ✓

Point 3 needs a small fix. The mixin's _forward stub is not about ordering — there's nothing that runs "before" the mixin, and it's not the mixin choosing not to execute something. The accurate picture is:
The mixin's _forward is a fallback that is only ever reached if a subclass forgot to define its own _forward. In normal operation — ResNet5 does define _forward — the mixin's _forward is completely shadowed and never executes at all. It's dead code in the happy path. It only becomes reachable when someone writes a subclass of the mixin but omits _forward; then self._forward falls through the MRO to the stub, which raises NotImplementedError to say "you were required to implement this and didn't."

In [21]:
from einops import rearrange

t = torch.randint(0,10,(2,1,10,64,64))

mutate_t = rearrange(t, "b c t h w -> (b t) c h w")
print(mutate_t.shape) 

unmutate_t = rearrange(mutate_t, "(b t) c h w -> b c t h w", b = 2)
print(unmutate_t.shape)

torch.Size([20, 1, 64, 64])
torch.Size([2, 1, 10, 64, 64])


In [38]:
avg_pool = nn.AdaptiveAvgPool2d((1,1))
t = torch.randn(128,256,32,32)
adaptive_output = avg_pool(t)
print(adaptive_output.shape)

adaptive_output.flatten(1).shape

torch.Size([128, 256, 1, 1])


torch.Size([128, 256])